# Execution notebook — Evaluation and experiment comparison

**Type:** execution notebook.

## Purpose

Compare Original and PP1–PP5 using `calculate_metrics()` with and without `pos_process()`. Write `04_pipeline_results/tabela_avaliacao_experiencias.csv`.

## Imports

Loads `postprocessing_common.ipynb` via `%run`.


## Configuration

In [ ]:
# ==========================================================
# EVALUATION CONFIGURATION
# ==========================================================

# Correct prediction orientation (as in Pos_Metrics.ipynb)
APPLY_ORIENTATION_CORRECTION = True

# Experiments to evaluate: (table label, results folder under 02_dataset/)
EXPERIMENTS_TO_EVALUATE = [
    ("Original", "results_original"),
    ("PP1", "results_pp_1"),
    ("PP2", "results_pp_2"),
    ("PP3", "results_pp_3"),
    ("PP4", "results_pp_4"),
    ("PP5", "results_pp_5"),
]

## Post-processing library (`%run`)

In [ ]:
%run ../03_postprocessing/postprocessing_common.ipynb

## Global validation (paths, datasets, pairing)

In [ ]:
import pandas as pd
from IPython.display import display

print("=" * 60)
print("PATH AND DATASET VALIDATION")
print("=" * 60)

warnings = validate_project_paths()
if warnings:
    for aviso in warnings:
        print(f"[AVISO] {aviso}")
else:
    print("Main folders found.")

print("\nPrediction ↔ label pairing (original identifier):")
pairing_report = validate_all_pairings()
for report_key, pairing_errors in pairing_report.items():
    if pairing_errors:
        print(f"  {report_key}: {len(pairing_errors)} error(s) — e.g. {pairing_errors[0]}")
    else:
        print(f"  {chave}: OK")


## Evaluate each experiment (with / without post-processing)

In [ ]:
def evaluate_experiment(experiment_name: str, results_folder_name: str) -> list:
    """
    Evaluates all predictions in a results folder.
    Returns two metric rows: without and with post-processing.
    """
    results_folder = DATA_DIR / results_folder_name
    predictions = list_predictions(results_folder)

    if len(predictions) == 0:
        return [
            {
                "Experiment": experiment_name,
                "Post_processing": "No",
                "N": 0,
                "Dice": np.nan,
                "Accuracy": np.nan,
                "Precision": np.nan,
                "Recall": np.nan,
                "Pasta": results_folder_name,
            },
            {
                "Experiment": experiment_name,
                "Post_processing": "Yes",
                "N": 0,
                "Dice": np.nan,
                "Accuracy": np.nan,
                "Precision": np.nan,
                "Recall": np.nan,
                "Pasta": results_folder_name,
            },
        ]

    metrics_without_postprocess = []
    metrics_with_postprocess = []

    for prediction_path in predictions:
        label_path = resolve_label_path(prediction_path.name)

        predicted = load_mask_png(prediction_path)
        gt = load_mask_png(label_path)

        if APPLY_ORIENTATION_CORRECTION:
            predicted = correct_prediction_orientation(predicted)

        # Metrics without post-processing (Pos_Metrics — primeiro bloco)
        dice, ac, pr, re = calculate_metrics(predicted, gt)
        metrics_without_postprocess.append((dice, ac, pr, re))

        # Post-processing + metrics (Pos_Metrics — segundo bloco)
        predicted_pos_proc = pos_process(predicted)
        dice, ac, pr, re = calculate_metrics(predicted_pos_proc, gt)
        metrics_with_postprocess.append((dice, ac, pr, re))

    mean_without_postprocess = compute_mean_metrics(metrics_without_postprocess)
    mean_with_postprocess = compute_mean_metrics(metrics_with_postprocess)

    return [
        {
            "Experiment": experiment_name,
            "Post_processing": "No",
            "N": len(predictions),
            "Dice": mean_without_postprocess["dice"],
            "Accuracy": mean_without_postprocess["accuracy"],
            "Precision": mean_without_postprocess["precision"],
            "Recall": mean_without_postprocess["recall"],
            "Pasta": results_folder_name,
        },
        {
            "Experiment": experiment_name,
            "Post_processing": "Yes",
            "N": len(predictions),
            "Dice": mean_with_postprocess["dice"],
            "Accuracy": mean_with_postprocess["accuracy"],
            "Precision": mean_with_postprocess["precision"],
            "Recall": mean_with_postprocess["recall"],
            "Pasta": results_folder_name,
        },
    ]


result_rows = []
for experiment_key, results_folder_key in EXPERIMENTS_TO_EVALUATE:
    result_rows.extend(evaluate_experiment(experiment_key, results_folder_key))

comparison_table = pd.DataFrame(result_rows)
column_order = [
    "Experiment",
    "Post_processing",
    "N",
    "Dice",
    "Accuracy",
    "Precision",
    "Recall",
    "Pasta",
]
comparison_table = comparison_table[column_order]

## Final comparison table

In [ ]:
pd.set_option("display.max_rows", 20)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

display(comparison_table)

output_csv_path = PROJECT_ROOT / "04_pipeline_results" / "tabela_avaliacao_experiencias.csv"
output_csv_path.parent.mkdir(parents=True, exist_ok=True)
comparison_table.to_csv(output_csv_path, index=False)
print(f"\nTable saved to: {output_csv_path}")

## Summary plot (Dice)

In [ ]:
import matplotlib.pyplot as plt

if len(comparison_table) > 0 and comparison_table["Dice"].notna().any():
    fig, ax = plt.subplots(figsize=(10, 5))
    for postprocess_flag, group_df in comparison_table.groupby("Post_processing"):
        ax.plot(
            group_df["Experiment"],
            group_df["Dice"],
            marker="o",
            label=f"Post-processing: {postprocess_flag}",
        )
    ax.set_ylabel("Mean Dice")
    ax.set_xlabel("Experiment")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## Conclusions

Comparison table and CSV saved under `04_pipeline_results/`. Use results in `05_report/`.
